# Customer Segmentation using K-Means Clustering
### Data Mining Project — RFM Analysis + Unsupervised Machine Learning

---

**Objective:** Group customers into distinct segments based on their purchasing behavior using K-Means Clustering and RFM Analysis.

**RFM stands for:**
- Recency — How recently did the customer buy?
- Frequency — How often do they buy?
- Monetary — How much do they spend?

**Pipeline Overview:**
1. Install and import libraries
2. Generate customer dataset
3. Exploratory Data Analysis
4. RFM Feature Engineering
5. Find optimal number of clusters
6. K-Means Clustering
7. Cluster Analysis and Profiling
8. Visualizations
9. Save results and model
10. Reusable prediction function

## Phase 1 — Install Required Libraries

We install all libraries needed for data processing, clustering, and visualization.

| Library | Purpose |
|---|---|
| pandas, numpy | Data manipulation |
| scikit-learn | K-Means clustering and preprocessing |
| matplotlib, seaborn | Visualizations |
| plotly | Interactive plots |
| yellowbrick | Elbow method visualization |

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn plotly yellowbrick --quiet
print('All libraries installed successfully.')

## Phase 2 — Import Libraries and Configuration

Import all required libraries and configure display settings. A fixed random seed ensures our results are reproducible every time we run the notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import os
import pickle

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('Libraries imported. Ready to go.')

## Phase 3 — Generate Customer Dataset

We simulate a realistic retail dataset with 5,000 customers containing purchase history, product preferences, and engagement data.

We intentionally design the data to have 4 natural customer groups that K-Means should discover on its own.

Note: If you have real data, replace this cell with pd.read_csv('your_file.csv') and map your column names accordingly.

In [ ]:
n = 5000
np.random.seed(42)

groups = {
    'Champions': int(n * 0.20),
    'Loyal'    : int(n * 0.30),
    'At Risk'  : int(n * 0.25),
    'Lost'     : int(n * 0.25),
}

dfs = []

n1 = groups['Champions']
dfs.append(pd.DataFrame({
    'customer_id'               : range(1, n1 + 1),
    'total_purchases'           : np.random.randint(30, 60, n1),
    'avg_order_value'           : np.round(np.random.uniform(300, 600, n1), 2),
    'days_since_last_purchase'  : np.random.randint(1, 30, n1),
    'total_spend'               : np.round(np.random.uniform(5000, 15000, n1), 2),
    'purchase_frequency_monthly': np.round(np.random.uniform(3, 6, n1), 2),
    'num_categories_bought'     : np.random.randint(6, 10, n1),
    'num_returns'               : np.random.randint(0, 2, n1),
    'true_segment'              : 'Champions'
}))

n2 = groups['Loyal']
dfs.append(pd.DataFrame({
    'customer_id'               : range(n1 + 1, n1 + n2 + 1),
    'total_purchases'           : np.random.randint(15, 35, n2),
    'avg_order_value'           : np.round(np.random.uniform(100, 300, n2), 2),
    'days_since_last_purchase'  : np.random.randint(20, 90, n2),
    'total_spend'               : np.round(np.random.uniform(1500, 5000, n2), 2),
    'purchase_frequency_monthly': np.round(np.random.uniform(1.5, 3.5, n2), 2),
    'num_categories_bought'     : np.random.randint(3, 7, n2),
    'num_returns'               : np.random.randint(0, 3, n2),
    'true_segment'              : 'Loyal'
}))

n3 = groups['At Risk']
dfs.append(pd.DataFrame({
    'customer_id'               : range(n1 + n2 + 1, n1 + n2 + n3 + 1),
    'total_purchases'           : np.random.randint(5, 20, n3),
    'avg_order_value'           : np.round(np.random.uniform(50, 150, n3), 2),
    'days_since_last_purchase'  : np.random.randint(90, 200, n3),
    'total_spend'               : np.round(np.random.uniform(500, 2000, n3), 2),
    'purchase_frequency_monthly': np.round(np.random.uniform(0.5, 1.5, n3), 2),
    'num_categories_bought'     : np.random.randint(2, 5, n3),
    'num_returns'               : np.random.randint(1, 4, n3),
    'true_segment'              : 'At Risk'
}))

n4 = groups['Lost']
dfs.append(pd.DataFrame({
    'customer_id'               : range(n1 + n2 + n3 + 1, n + 1),
    'total_purchases'           : np.random.randint(1, 8, n4),
    'avg_order_value'           : np.round(np.random.uniform(20, 80, n4), 2),
    'days_since_last_purchase'  : np.random.randint(200, 365, n4),
    'total_spend'               : np.round(np.random.uniform(50, 600, n4), 2),
    'purchase_frequency_monthly': np.round(np.random.uniform(0.1, 0.5, n4), 2),
    'num_categories_bought'     : np.random.randint(1, 3, n4),
    'num_returns'               : np.random.randint(0, 2, n4),
    'true_segment'              : 'Lost'
}))

df = pd.concat(dfs, ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

os.makedirs('data/raw', exist_ok=True)
df.to_csv('data/raw/customer_data.csv', index=False)

print(f'Dataset created: {df.shape[0]} rows x {df.shape[1]} columns')
print(f"\nCustomer group distribution:\n{df['true_segment'].value_counts()}")
df.head()

## Phase 4 — Exploratory Data Analysis

Before clustering, we explore the data to understand feature distributions, relationships between variables, and any outliers that could affect clustering results.

Good EDA helps us choose the right features and preprocessing steps for K-Means.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

features_to_plot = [
    ('total_purchases', '#3498db'),
    ('avg_order_value', '#2ecc71'),
    ('days_since_last_purchase', '#e74c3c'),
    ('total_spend', '#9b59b6'),
    ('purchase_frequency_monthly', '#f39c12'),
    ('num_categories_bought', '#1abc9c')
]

for ax, (feat, color) in zip(axes.flatten(), features_to_plot):
    ax.hist(df[feat], bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')

plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
colors = {'Champions': '#2ecc71', 'Loyal': '#3498db', 'At Risk': '#f39c12', 'Lost': '#e74c3c'}
for segment, color in colors.items():
    subset = df[df['true_segment'] == segment]
    plt.scatter(subset['total_spend'], subset['purchase_frequency_monthly'],
                alpha=0.4, label=segment, color=color, s=15)
plt.xlabel('Total Spend')
plt.ylabel('Purchase Frequency (Monthly)')
plt.title('Total Spend vs Purchase Frequency by Segment', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
num_cols = df.select_dtypes(include=np.number).drop('customer_id', axis=1)
sns.heatmap(num_cols.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 9})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Summary Statistics:')
df.describe().T

## Phase 5 — RFM Feature Engineering

RFM is the most widely used framework for customer segmentation in retail data mining.

| Feature | Calculation | What It Captures |
|---|---|---|
| Recency Score | 1 - (days_since / max_days) | Higher = more recent buyer |
| Frequency Score | purchases / max_purchases | Higher = buys more often |
| Monetary Score | spend / max_spend | Higher = spends more |
| RFM Combined | Average of all 3 | Overall customer value |
| Return Rate | returns / purchases | Higher = lower loyalty |

All scores are normalized between 0 and 1 so that no single feature dominates the clustering.

In [ ]:
df['recency_score']   = 1 - (df['days_since_last_purchase'] / df['days_since_last_purchase'].max())
df['frequency_score'] = df['total_purchases'] / df['total_purchases'].max()
df['monetary_score']  = df['total_spend'] / df['total_spend'].max()
df['rfm_combined']    = (df['recency_score'] + df['frequency_score'] + df['monetary_score']) / 3
df['return_rate']     = df['num_returns'] / (df['total_purchases'] + 1)

os.makedirs('data/processed', exist_ok=True)
df.to_csv('data/processed/customer_data_rfm.csv', index=False)

print('RFM features created: recency_score, frequency_score, monetary_score, rfm_combined, return_rate')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feature in zip(axes, ['recency_score', 'frequency_score', 'monetary_score']):
    for segment in ['Champions', 'Loyal', 'At Risk', 'Lost']:
        subset = df[df['true_segment'] == segment][feature]
        ax.hist(subset, bins=20, alpha=0.5, label=segment)
    ax.set_title(feature.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('RFM Score Distributions by Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Phase 6 — Find Optimal Number of Clusters

K-Means requires us to specify K (number of clusters) before running. We use two methods to find the best K:

- Elbow Method: Plot inertia vs K. The point where the curve bends like an elbow is the optimal K.
- Silhouette Score: Measures how well-separated the clusters are. Higher is better, with a maximum of 1.0.

Both methods should point to the same K value for a reliable result.

In [ ]:
CLUSTER_FEATURES = [
    'recency_score', 'frequency_score', 'monetary_score',
    'avg_order_value', 'num_categories_bought', 'return_rate'
]

X        = df[CLUSTER_FEATURES].copy()
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias   = []
sil_scores = []
K_range    = range(2, 11)

for k in K_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].axvline(x=4, color='red', linestyle='--', label='Optimal K = 4')
axes[0].legend()

axes[1].plot(K_range, sil_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score', fontsize=13, fontweight='bold')
axes[1].axvline(x=4, color='red', linestyle='--', label='Optimal K = 4')
axes[1].legend()

plt.tight_layout()
plt.show()

optimal_k = K_range[sil_scores.index(max(sil_scores))]
print(f'Optimal number of clusters: K = {optimal_k}')
print(f'Best Silhouette Score: {max(sil_scores):.4f}')

## Phase 7 — K-Means Clustering

Now we apply K-Means with our optimal K value.

How K-Means works:
1. Randomly place K centroids (cluster centers)
2. Assign each customer to the nearest centroid
3. Move each centroid to the average of its assigned customers
4. Repeat steps 2 and 3 until centroids stop moving

The result is K groups where customers within the same group are as similar as possible and customers across groups are as different as possible.

In [ ]:
K      = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10, max_iter=300)
df['cluster'] = kmeans.fit_predict(X_scaled)

final_sil = silhouette_score(X_scaled, df['cluster'])
print(f'K-Means clustering complete.')
print(f'Number of Clusters : {K}')
print(f'Silhouette Score   : {final_sil:.4f}')
print(f'\nCluster sizes:')
print(df['cluster'].value_counts().sort_index())

cluster_means = df.groupby('cluster')[['recency_score', 'frequency_score', 'monetary_score', 'rfm_combined']].mean()
print('\nCluster RFM Averages:')
print(cluster_means.round(3))

## Phase 8 — Cluster Profiling and Naming

Raw cluster numbers (0, 1, 2, 3) are not meaningful on their own. We analyze each cluster's average RFM values and assign business-friendly names.

This step turns a mathematical result into a business insight that stakeholders can understand and act on.

| Segment | Behavior | Recommended Action |
|---|---|---|
| Champions | High spend, frequent, recent | Reward with loyalty program |
| Loyal Customers | Regular buyers, medium spend | Upsell premium products |
| At Risk | Used to buy, now inactive | Send win-back offers |
| Lost Customers | Long inactive, low spend | Aggressive re-engagement campaign |

In [ ]:
profile_cols = [
    'total_purchases', 'avg_order_value', 'days_since_last_purchase',
    'total_spend', 'purchase_frequency_monthly', 'rfm_combined'
]
profile = df.groupby('cluster')[profile_cols].mean().round(2)
print('Cluster Profiles:')
print(profile)

rfm_rank     = df.groupby('cluster')['rfm_combined'].mean().rank(ascending=False)
segment_map  = {}
name_by_rank = {1: 'Champions', 2: 'Loyal Customers', 3: 'At Risk', 4: 'Lost Customers'}
for cluster_id, rank in rfm_rank.items():
    segment_map[cluster_id] = name_by_rank[int(rank)]

df['segment'] = df['cluster'].map(segment_map)

print('\nSegment mapping:')
for k, v in segment_map.items():
    print(f'   Cluster {k} -> {v}')

print('\nFinal segment distribution:')
print(df['segment'].value_counts())

print('\nSegment Summary:')
df.groupby('segment')[profile_cols].mean().round(2)

## Phase 9 — Cluster Visualizations

We visualize the clusters from multiple angles:

- Bar chart: Segment sizes
- PCA scatter plot: 2D view of all clusters using dimensionality reduction
- Box plots: RFM score distribution per segment

PCA (Principal Component Analysis) reduces our 6 clustering features down to 2 dimensions so we can plot all customers on a single chart while preserving the cluster structure.

In [ ]:
seg_colors = {
    'Champions'      : '#2ecc71',
    'Loyal Customers': '#3498db',
    'At Risk'        : '#f39c12',
    'Lost Customers' : '#e74c3c'
}

# Segment Size Bar Chart
seg_counts = df['segment'].value_counts()
plt.figure(figsize=(9, 5))
bars = plt.bar(seg_counts.index, seg_counts.values,
               color=[seg_colors[s] for s in seg_counts.index], edgecolor='black')
for bar, val in zip(bars, seg_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             str(val), ha='center', fontweight='bold')
plt.title('Customer Segment Sizes', fontsize=14, fontweight='bold')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()

# PCA 2D Scatter Plot
pca        = PCA(n_components=2, random_state=42)
X_pca      = pca.fit_transform(X_scaled)
df['pca1'] = X_pca[:, 0]
df['pca2'] = X_pca[:, 1]

plt.figure(figsize=(10, 7))
for segment, color in seg_colors.items():
    subset = df[df['segment'] == segment]
    plt.scatter(subset['pca1'], subset['pca2'],
                label=segment, color=color, alpha=0.4, s=15)
plt.xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Customer Segments — PCA View', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

# RFM Box Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
rfm_features = ['recency_score', 'frequency_score', 'monetary_score']
for ax, feat in zip(axes, rfm_features):
    data_by_seg = [df[df['segment'] == seg][feat].values for seg in seg_colors.keys()]
    bp = ax.boxplot(data_by_seg, patch_artist=True, labels=list(seg_colors.keys()))
    for patch, color in zip(bp['boxes'], seg_colors.values()):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('RFM Scores by Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('All visualizations generated.')

## Phase 10 — Save Results and Model

We save the following files:

- customer_segments.csv: Full dataset with cluster labels assigned to each customer
- kmeans_model.pkl: Trained K-Means model
- scaler.pkl: Feature scaler used during training
- segment_map.pkl: Mapping from cluster number to segment name
- cluster_features.pkl: List of feature names used for clustering

These files allow you to apply the model to new customers in the future without retraining from scratch.

In [ ]:
os.makedirs('models', exist_ok=True)

output_cols = ['customer_id', 'total_purchases', 'avg_order_value',
               'days_since_last_purchase', 'total_spend',
               'rfm_combined', 'cluster', 'segment']
df[output_cols].to_csv('data/processed/customer_segments.csv', index=False)

with open('models/kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('models/segment_map.pkl', 'wb') as f:
    pickle.dump(segment_map, f)

with open('models/cluster_features.pkl', 'wb') as f:
    pickle.dump(CLUSTER_FEATURES, f)

print('All files saved.')
print('\n   data/processed/customer_segments.csv')
print('   models/kmeans_model.pkl')
print('   models/scaler.pkl')
print('   models/segment_map.pkl')
print('   models/cluster_features.pkl')

print('\nFinal Segment Summary:')
summary = df.groupby('segment').agg(
    num_customers    = ('customer_id', 'count'),
    avg_spend        = ('total_spend', 'mean'),
    avg_frequency    = ('purchase_frequency_monthly', 'mean'),
    avg_recency_days = ('days_since_last_purchase', 'mean'),
    avg_rfm          = ('rfm_combined', 'mean')
).round(2)
print(summary)

## Phase 11 — Reusable Prediction Function

This function takes any new customer's data and returns their segment label along with a recommended marketing action.

In a real business setting, this function would be called every time a new customer is added to the database, automatically assigning them to a segment so the marketing team knows how to engage them.

In [ ]:
def predict_customer_segment(customer: dict) -> dict:
    """
    Predict the segment for a new customer.

    Args:
        customer (dict): Customer features as a dictionary.

    Returns:
        dict: cluster number, segment name, and recommended action.
    """
    with open('models/kmeans_model.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('models/scaler.pkl', 'rb') as f:
        sc = pickle.load(f)
    with open('models/segment_map.pkl', 'rb') as f:
        seg_map = pickle.load(f)
    with open('models/cluster_features.pkl', 'rb') as f:
        features = pickle.load(f)

    actions = {
        'Champions'      : 'Reward with loyalty program and exclusive offers',
        'Loyal Customers': 'Upsell premium products and membership plans',
        'At Risk'        : 'Send personalized win-back email with discount',
        'Lost Customers' : 'Send aggressive re-engagement campaign'
    }

    X    = pd.DataFrame([customer])[features]
    X_sc = sc.transform(X)
    cluster = model.predict(X_sc)[0]
    segment = seg_map[cluster]

    return {
        'cluster' : int(cluster),
        'segment' : segment,
        'action'  : actions[segment]
    }


# Test with two sample customers
sample_champion = {
    'recency_score'        : 0.95,
    'frequency_score'      : 0.85,
    'monetary_score'       : 0.90,
    'avg_order_value'      : 450.0,
    'num_categories_bought': 8,
    'return_rate'          : 0.02
}

sample_lost = {
    'recency_score'        : 0.10,
    'frequency_score'      : 0.05,
    'monetary_score'       : 0.04,
    'avg_order_value'      : 35.0,
    'num_categories_bought': 1,
    'return_rate'          : 0.20
}

print('Sample Predictions:\n')
for label, customer in [('High-value customer', sample_champion), ('Inactive customer', sample_lost)]:
    result = predict_customer_segment(customer)
    print(f'{label}:')
    print(f"   Segment : {result['segment']}")
    print(f"   Action  : {result['action']}")
    print()

---

## Project Complete

| Phase | What We Did |
|---|---|
| 1 | Installed all libraries |
| 2 | Imported and configured environment |
| 3 | Generated realistic 5,000-customer dataset |
| 4 | Explored data with EDA and visualizations |
| 5 | Engineered RFM features for data mining |
| 6 | Found optimal K using Elbow and Silhouette methods |
| 7 | Applied K-Means clustering |
| 8 | Profiled and named each customer segment |
| 9 | Visualized clusters with PCA and box plots |
| 10 | Saved model, scaler, and segmented data |
| 11 | Built reusable segment prediction function |

---

## Next Steps (Optional)

- Try DBSCAN or Hierarchical Clustering and compare results with K-Means
- Add a time-series component to track how customers move between segments over months
- Deploy as a Streamlit dashboard for real-time customer lookup